## Why MongoDB is useful here

NorthStar has app events, complaints, incidents, and case history that can grow and change over time.
A document model is useful because related records can stay together in one order case document.

In [ ]:
import json
from pathlib import Path

import pandas as pd

ROOT_DIR = Path('/content/assignment')

In [ ]:
customers = pd.read_csv(ROOT_DIR / 'customers.csv')
orders = pd.read_csv(ROOT_DIR / 'orders.csv')
deliveries = pd.read_csv(ROOT_DIR / 'deliveries.csv')
complaints = pd.read_csv(ROOT_DIR / 'complaints.csv')
incidents = pd.read_csv(ROOT_DIR / 'incidents.csv')
app_events = pd.read_csv(ROOT_DIR / 'app_events.csv')
drivers = pd.read_csv(ROOT_DIR / 'drivers.csv')
vehicles = pd.read_csv(ROOT_DIR / 'vehicles.csv')
hubs = pd.read_csv(ROOT_DIR / 'hubs.csv')

In [ ]:
customer_lookup = customers.set_index('customer_id')
delivery_lookup = deliveries.set_index('order_id')
driver_lookup = drivers.set_index('driver_id')
vehicle_lookup = vehicles.set_index('vehicle_id')
hub_lookup = hubs.set_index('hub_id')

def safe_value(value):
    if pd.isna(value):
        return None
    return value

def row_to_dict(row, columns):
    return {column: safe_value(row[column]) for column in columns if column in row.index}

In [ ]:
sample_order = orders.iloc[0]
order_id = sample_order['order_id']
customer_id = sample_order['customer_id']

customer_row = customer_lookup.loc[customer_id]
delivery_row = delivery_lookup.loc[order_id] if order_id in delivery_lookup.index else None

driver_row = driver_lookup.loc[delivery_row['driver_id']] if delivery_row is not None else None
vehicle_row = vehicle_lookup.loc[delivery_row['vehicle_id']] if delivery_row is not None else None
hub_row = hub_lookup.loc[delivery_row['hub_id']] if delivery_row is not None else None

document = {
    '_id': order_id,
    'customer': row_to_dict(customer_row, ['customer_id', 'age', 'home_zone', 'customer_type', 'account_status']),
    'order': row_to_dict(sample_order, ['order_id', 'service_type', 'pickup_zone', 'dropoff_zone', 'priority_level', 'order_value']),
    'delivery': row_to_dict(delivery_row, ['delivery_id', 'driver_id', 'vehicle_id', 'hub_id', 'delivery_status', 'manual_route_override_count', 'customer_rating_post_delivery']) if delivery_row is not None else None,
    'driver': row_to_dict(driver_row, ['driver_id', 'base_zone', 'employment_type', 'driver_rating']) if driver_row is not None else None,
    'vehicle': row_to_dict(vehicle_row, ['vehicle_id', 'vehicle_type', 'assigned_zone', 'maintenance_status']) if vehicle_row is not None else None,
    'hub': row_to_dict(hub_row, ['hub_id', 'hub_name', 'zone', 'hub_type']) if hub_row is not None else None,
    'complaints': complaints[complaints['order_id'] == order_id].to_dict(orient='records'),
    'incidents': incidents[incidents['delivery_id'] == delivery_row['delivery_id']].to_dict(orient='records') if delivery_row is not None else [],
    'app_events': app_events[app_events['order_id'] == order_id].to_dict(orient='records')
}

print(json.dumps(document, indent=2, default=str))

## Indexes to explain in the report

Useful indexes:
- `customer.customer_id`
- `order.service_type`
- `delivery.delivery_status`

In [ ]:
# Optional Atlas upload example
# !pip install pymongo
# from pymongo import MongoClient, ReplaceOne
# client = MongoClient('your_mongodb_connection_string')
# collection = client['northstar']['order_cases']
# collection.create_index('customer.customer_id')
# collection.create_index('order.service_type')
# collection.create_index('delivery.delivery_status')

## What to write after this

Explain:
- why one order case document is useful
- why complaints and app events fit well as arrays
- how this design supports faster case investigation